# CALLIC — continue training from checkpoint
Quick setup: locate repo → fetch weights → verify → relaunch with `--resume` → poll.
Run cells top to bottom. Re-run the last cell anytime to check progress.

In [ ]:
import os, sys
REPO_URL = ''  # git URL alternative — only needed if the repo isn't uploaded
DRIVE_FOLDER_ID = '1rjJWK88H6OuasgYEQf_ujbNshKtZ7lMO'  # 'CALLIC repo' Drive folder
def _find_repo():
    cands = [os.getcwd(), '/content']
    try:
        import glob as _g
        cands += sorted(_g.glob('/content/drive/MyDrive/*/callic'))
    except Exception:
        pass
    for c in cands:
        root = c if os.path.basename(c) != 'callic' else os.path.dirname(c)
        if os.path.isdir(os.path.join(root, 'callic')):
            return root
    return None
ROOT = _find_repo()
if ROOT is None:
    import subprocess
    if DRIVE_FOLDER_ID:
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'gdown'])
        import gdown
        gdown.download_folder(f'https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}',
                              output='/content/callic', quiet=False)
        ROOT = '/content/callic'
    elif REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, '/content/repo'], check=True)
        ROOT = '/content/repo'
    else:
        raise SystemExit('repo not found: upload it via Files panel or set REPO_URL and re-run')
    assert os.path.isdir(os.path.join(ROOT, 'callic')), f'fetched dir lacks callic/ (got {os.listdir(ROOT)[:10]})'
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print('repo root:', ROOT)
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())


In [ ]:
# Checkpoint: local checkpoints/ first, else Drive link, else Drive mount.
# Drive file (74k-step MGCF + best@60500):
# https://drive.google.com/file/d/1rvVllv3Numgt_RyfBssTcJhZcmavOm6q/view?usp=sharing
import glob, subprocess, zipfile
FILE_ID = '1rvVllv3Numgt_RyfBssTcJhZcmavOm6q'
os.makedirs('checkpoints', exist_ok=True)
def _try_load(path):
    try:
        q = torch.load(path, map_location='cpu')
        return q if isinstance(q, dict) and 'model' in q else None
    except Exception:
        return None
cands = sorted(glob.glob('checkpoints/*.pt'))
if not cands:
    for mnt in glob.glob('/content/drive/MyDrive/callic/run_100k/ckpts/*.pt'):
        cands.append(mnt)
if not cands:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'gdown'])
    import gdown
    dl = 'checkpoints/_drive_dl.bin'
    gdown.download(f'https://drive.google.com/uc?id={FILE_ID}', dl, quiet=False)
    if _try_load(dl) is not None:
        os.rename(dl, 'checkpoints/mgcf_drive.pt')
    else:
        assert open(dl, 'rb').read(2) == b'PK', 'download is neither ckpt nor zip'
        with zipfile.ZipFile(dl) as z:
            z.extractall('drive_pull')
        os.remove(dl)
    cands = sorted(glob.glob('checkpoints/*.pt') + glob.glob('drive_pull/**/*.pt', recursive=True))
print('ckpts:', cands)
# prefer latest-step file; override RESUME_CKPT manually if you want best/numbered
best = (None, -1)
for f in cands:
    q = _try_load(f)
    if q and int(q.get('step', -1)) > best[1]:
        best = (f, int(q.get('step', -1)))
RESUME_CKPT = best[0]
q = _try_load(RESUME_CKPT)
print(f"RESUME_CKPT={RESUME_CKPT} | step={q.get('step')} | best={q.get('best_loss')}@{q.get('best_step')}")


In [ ]:
# Training data: quick default is DIV2K-valid (449MB). Full paper data: bash tools/lightning_setup.sh
import subprocess
from pathlib import Path
DATA = 'data/DIV2K_valid_HR'
if len(list(Path(DATA).glob('*.png'))) < 100:
    Path(DATA).mkdir(parents=True, exist_ok=True)
    subprocess.run('cd data && curl -L -o DIV2K_valid_HR.zip https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip && unzip -q -o DIV2K_valid_HR.zip && rm DIV2K_valid_HR.zip', shell=True)
print('patches source:', DATA, len(list(Path(DATA).glob('*.png'))), 'images')


In [ ]:
# Relaunch training in background from RESUME_CKPT (safe to re-run: --resume continues exact step).
# Tune --steps/--bs/--lr for your GPU; defaults continue the 100k run config.
import subprocess
OUT = RESUME_CKPT  # checkpoints update in place; numbered/best keeps preserved alongside
cmd = (f'{sys.executable} tools/train.py --data {DATA} --steps 100000 --bs 32 --lr 5e-4 '
       f'--schedule cosine --log-every 500 --keep-every 10000 --keep-last 3 '
       f'--resume --out {OUT} > train.log 2>&1 & echo $!')
print(subprocess.run(f'nohup {cmd}', shell=True, capture_output=True, text=True).stdout)
print('launched. watch with: tail -f train.log')


In [ ]:
# Poll progress (re-run anytime). Shows: process alive?, log tail, ckpt files.
import subprocess, time
print('wall:', time.strftime('%Y-%m-%d %H:%M:%S'))
print(subprocess.run(['pgrep', '-af', 'tools/train.py'], capture_output=True, text=True).stdout or '(trainer not running)')
print('--- train.log tail ---')
print(subprocess.run(['tail', '-5', 'train.log'], capture_output=True, text=True).stdout or '(no log yet)')
print('--- checkpoints ---')
print(subprocess.run(['ls', '-la', 'checkpoints/'], capture_output=True, text=True).stdout)
print(subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() or '(no nvidia-smi)')
